# Apriori Algorithm 

## 1. What is the Apriori Algorithm?
The **Apriori Algorithm** is an unsupervised learning algorithm used for **Association Rule Mining**. It is most famously used for **Market Basket Analysis**—discovering how items are purchased together (e.g., "Customers who bought diapers also bought beer").

* **The Core Principle (The Apriori Property):** If an itemset is frequent, then *all of its subsets must also be frequent*. 
* **The Pruning Step:** Conversely, if a subset is infrequent, the superset cannot be frequent. The algorithm uses this rule to dramatically reduce the number of combinations it needs to check, saving massive computational power.



---

## 2. The 3 Core Metrics 
To evaluate how strong an association rule ($A \rightarrow B$) is, Apriori relies on three mathematical metrics:

### A. Support (How Popular is the Item?)
Measures the frequency of an item or itemset in the entire dataset.
* **Formula:**
  $$\text{Support}(A) = \frac{\text{Transactions containing } A}{\text{Total Transactions}}$$
* **Example:** If 100 people visit a store and 20 buy bread, the support for Bread is 20%.

### B. Confidence (How Likely is $B$ given $A$?)
Measures the likelihood that item $B$ is purchased when item $A$ is purchased. It is a directional metric ($A \rightarrow B$ is not the same as $B \rightarrow A$).
* **Formula:**
  $$\text{Confidence}(A \rightarrow B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$$
* **Example:** Out of the 20 people who bought Bread, 10 also bought Butter. The Confidence of (Bread $\rightarrow$ Butter) is 50%.

### C. Lift (Does $A$ actually drive $B$?)
Measures how much the likelihood of buying $B$ increases when $A$ is bought, compared to $B$'s normal popularity. This controls for items that are just universally popular (like milk).
* **Formula:**
  $$\text{Lift}(A \rightarrow B) = \frac{\text{Confidence}(A \rightarrow B)}{\text{Support}(B)}$$
* **Interpretation:**
  * **Lift = 1:** No association (A and B are independent).
  * **Lift > 1:** Positive association (Buying A increases the chance of buying B).
  * **Lift < 1:** Negative association (Buying A decreases the chance of buying B).

---

## 3. How Apriori Works (Step-by-Step)
Apriori uses a "bottom-up" approach, generating candidates layer by layer.

1. **Set a Minimum Support & Confidence:** The user defines thresholds (e.g., `min_support = 0.05`, `min_confidence = 0.6`).
2. **1-Itemsets:** Calculate the support for all individual items. Drop any items that fall below `min_support`.
3. **2-Itemsets:** Combine the surviving items into pairs. Calculate their support. Drop pairs below `min_support`.
4. **Repeat (K-Itemsets):** Continue combining surviving itemsets into groups of 3, then 4, etc., until no more valid combinations can be formed.
5. **Generate Rules:** For the surviving frequent itemsets, calculate the **Confidence** and **Lift** to form the final association rules.



---

## 4. Python Implementation (`mlxtend`)
*Note: Scikit-Learn does NOT have a native Apriori algorithm. The industry standard library for this is `mlxtend`.*

```python
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. The data must be One-Hot Encoded (Rows = Transactions, Columns = Items)
# True/False or 1/0 indicating if the item was in the transaction
X_encoded = pd.DataFrame(...) 

# 2. Find frequent itemsets using the Apriori algorithm
frequent_itemsets = apriori(
    X_encoded, 
    min_support=0.05,  # Itemsets must appear in 5% of transactions
    use_colnames=True  # Keeps the actual item names instead of column indices
)

# 3. Generate association rules based on a metric (e.g., Lift)
rules = association_rules(
    frequent_itemsets, 
    metric="lift", 
    min_threshold=1.2  # Only keep rules with a lift > 1.2
)

# Sort the rules to see the strongest associations first
rules = rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])


```


---

## 5. Advantages & Disadvantages
- Advantages
 - Extremely easy to understand, implement, and explain to business stakeholders.

 - The "Apriori Property" pruning drastically reduces the math required compared to brute-forcing all combinations.

 - Does not require labeled data (Unsupervised).

- Disadvantages
 - **Computationally Expensive:** Even with pruning, generating candidate sets for massive databases (like Amazon's catalog) requires immense memory and time.

 - **Multiple Scans:** It has to scan the entire database multiple times (once for every step/layer), which is highly inefficient for large datasets.

## 6. Apriori vs. FP-Growth
If Apriori is too slow, the modern alternative is FP-Growth (Frequent Pattern Growth).

| Feature                       | Apriori                                      | FP-Growth                               |
| ----------------------------- | -------------------------------------------- | --------------------------------------- |
| **Candidate Generation**      | Yes (Generates many candidate itemsets)      | No (Uses compressed FP-Tree instead)    |
| **Database Scans**            | Multiple scans (for every level of itemsets) | Only **2 scans** of the database        |
| **Memory Usage**              | Very High                                    | Low                                     |
| **Speed**                     | Slow                                         | Very Fast                               |
| **Algorithm Type**            | Breadth-first search approach                | Tree-based divide-and-conquer           |
| **Data Structure Used**       | Candidate itemset lists                      | FP-Tree (Frequent Pattern Tree)         |
| **Performance on Large Data** | Poor performance                             | Works well with large datasets          |
| **Main Idea**                 | Generate candidates → test support           | Compress data → mine patterns from tree |
| **Best Use Case**             | Small datasets                               | Large datasets with many transactions   |


**Apriori** → Easy to understand but slow and memory expensive.

**FP-Growth** → Faster and more scalable because it avoids candidate generation.


